# NSE Stock Analysis

This notebook loads the processed stock data with technical metrics.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv('nse_stock_data_with_metrics.csv')

# Convert Date to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Display the first few rows
df.head()

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Check data info
df.info()

In [ ]:
last_3_dates = sorted(df['Date'].unique())[-3:]
recent_df=df[df['Date'].isin(last_3_dates)]
filtered_stocks = recent_df[
    ((recent_df['20VWMA'] - recent_df['20DMA']) / recent_df['20DMA'] > 0.02) & 
    (recent_df['200DMA_LINE_ANGLE'] > 10)
]['Ticker'].unique()
recent_df.describe()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Ensure Date is in datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Identify the last 3 unique dates in the dataset
last_3_dates = sorted(df['Date'].unique())[-3:]
recent_df = df[df['Date'].isin(last_3_dates)]

# Filter stocks: 20VWMA > 20dma by > 2% and 200DMA_LINE_ANGLE > 10
filtered_stocks = recent_df[
    ((recent_df['20VWMA'] - recent_df['20DMA']) / recent_df['20DMA'] > 0.02) & 
    (recent_df['200DMA_LINE_ANGLE'] > 10)
]['Ticker'].unique()

print(f"Stocks identified: {list(filtered_stocks)}")

for stock in filtered_stocks:
    # Get historical data for the stock (e.g., last 120 days for context)
    stock_df = df[df['Ticker'] == stock].sort_values('Date').tail(120)
    
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True, gridspec_kw={'height_ratios': [3, 1, 1]})
    fig.suptitle(f"Technical Analysis: {stock}", fontsize=16, fontweight='bold')

    # Plot 1: Price, 20 DMA, 200 DMA, and 20 VWMA
    ax1.plot(stock_df['Date'], stock_df['Close'], label='Price', color='black', linewidth=1.5)
    ax1.plot(stock_df['Date'], stock_df['20DMA'], label='20 DMA', color='blue', linestyle='--')
    ax1.plot(stock_df['Date'], stock_df['200DMA'], label='200 DMA', color='red', linewidth=2)
    ax1.plot(stock_df['Date'], stock_df['20VWMA'], label='20 VWMA', color='orange', linestyle='-.')
    ax1.set_ylabel('Price')
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.3)

    # Plot 2: Volume
    ax2.bar(stock_df['Date'], stock_df['Volume'], color='teal', alpha=0.6)
    ax2.set_ylabel('Volume')
    ax2.grid(True, alpha=0.3)

    # Plot 3: RSI
    ax3.plot(stock_df['Date'], stock_df['RSI_14'], label='RSI', color='purple')
    ax3.axhline(70, color='red', linestyle='--', alpha=0.5)
    ax3.axhline(30, color='green', linestyle='--', alpha=0.5)
    ax3.set_ylabel('RSI')
    ax3.set_ylim(0, 100)
    ax3.grid(True, alpha=0.3)

    plt.xticks(rotation=45)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Identify stocks meeting the criteria
def find_crossover_stocks(df):
    # Ensure data is sorted by symbol and date
    df = df.sort_values(['Ticker', 'Date'])
    
    # Calculate crossover: 20DMA was below 200DMA and is now above
    df['prev_20DMA'] = df.groupby('Ticker')['20DMA'].shift(10)
    df['prev_200DMA'] = df.groupby('Ticker')['200DMA'].shift(10)
    
    df['is_crossover'] = (df['20DMA'] > df['200DMA']) & (df['prev_20DMA'] <= df['prev_200DMA'])
    
    # Filter for criteria: crossover in last 3 days, 20DMA angle > 50, 200DMA angle > 5
    recent_df = df.groupby('Ticker').tail(20)
    matches = recent_df[
        (recent_df['is_crossover'] == True) & 
        (recent_df['20DMA_LINE_ANGLE'] -recent_df['200DMA_LINE_ANGLE']> 20) & 
        (recent_df['200DMA_LINE_ANGLE'] > 5)
    ]
    print(matches['Ticker'].unique() )
    return matches['Ticker'].unique()

identified_stocks = find_crossover_stocks(df)
df['Date'] = pd.to_datetime(df['Date'])
df = df[df['Date'] >= df['Date'].max() - pd.DateOffset(months=3)]

# 2. Conduct Technical Analysis and Display Charts
for stock in identified_stocks:
    stock_data = df[df['Ticker'] == stock].tail(100) # Look at last 100 days for context
    
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True, 
                                        gridspec_kw={'height_ratios': [3, 1, 1]})
    
    # Price and Moving Averages
    ax1.plot(stock_data['Date'], stock_data['Close'], label='Price', color='black', alpha=0.6)
    ax1.plot(stock_data['Date'], stock_data['20DMA'], label='20 DMA', color='blue')
    ax1.plot(stock_data['Date'], stock_data['200DMA'], label='200 DMA', color='red')
    ax1.set_title(f"Technical Analysis for {stock}")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Volume
    ax2.bar(stock_data['Date'], stock_data['Volume'], color='gray', alpha=0.5)
    ax2.set_ylabel('Volume')
    ax2.grid(True, alpha=0.3)

    # RSI
    ax3.plot(stock_data['Date'], stock_data['RSI_14'], color='purple', label='RSI (14)')
    ax3.axhline(70, color='red', linestyle='--', alpha=0.5)
    ax3.axhline(30, color='green', linestyle='--', alpha=0.5)
    ax3.set_ylabel('RSI')
    ax3.set_ylim(0, 100)
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
